# 03 · Camada Gold — Modelagem Dimensional (Star Schema)

Constrói as dimensões, a fato e as tabelas-ponte (*bridge*) a partir da Silver.

**Decisões de modelagem:**
- **Surrogate keys** `BIGINT` via `row_number()` ordenado pela chave natural (determinístico e sem colisões, ao contrário de hash truncado). Para o volume do projeto (~100 mil filmes) o custo da janela global é irrelevante.
- **Fato com grão = 1 filme** e somente filmes **Lançados** (conforme enunciado: "consolidar métricas de filmes lançados"). Os joins com financeiro/engajamento são 1:1 (Silver já deduplicada por filme) e há uma **asserção de unicidade** de grão.
- `dim_people` é única por **(nome, tipo)**: a mesma pessoa pode ser Ator e Diretor, e o `tipo_pessoa` fica na dimensão (as pontes só ligam filme↔pessoa).
- **Pontes** M:N (`genre`, `person`, `company`) evitam duplicar linhas da fato.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalogo", "workspace", "Catálogo (Unity Catalog)")
CATALOGO = dbutils.widgets.get("catalogo")

spark.sql(f"USE CATALOG `{CATALOGO}`")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold COMMENT 'Camada Gold: Star Schema para BI/IA'")


def gravar_gold(df, tabela):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    print(f"{tabela:34s} linhas={spark.table(tabela).count():>9,}")


def surrogate_key(ordem):
    """Chave substituta sequencial BIGINT, determinística (ordenada pela chave natural)."""
    return F.row_number().over(Window.orderBy(*ordem)).cast("bigint")

## Dimensões

In [ ]:
# ---------------- gold.dim_movies ----------------
dim_movies = (
    spark.table("silver.tb_info_filmes")
    .select(
        surrogate_key([F.col("id_filme")]).alias("sk_movie_id"),
        F.col("id_filme").cast("string").alias("id_filme"),
        "titulo",
        F.col("data_lancamento").cast("date").alias("data_lancamento"),
        F.col("ano_lancamento").cast("int").alias("ano_lancamento"),
        F.col("duracao_minutos").cast("int").alias("duracao_minutos"),
        "idioma_original",
        "status_filme",
        "sinopse",
    )
)
gravar_gold(dim_movies, "gold.dim_movies")

# ---------------- gold.dim_genres ----------------
dim_genres = (
    spark.table("silver.tb_generos").select(F.col("genero").alias("nome_genero")).distinct()
    .select(surrogate_key([F.col("nome_genero")]).alias("sk_genre_id"), "nome_genero")
)
gravar_gold(dim_genres, "gold.dim_genres")

# ---------------- gold.dim_people (Ator, Diretor, Roteirista) ----------------
dim_people = (
    spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa")).distinct()
    .select(surrogate_key([F.col("tipo_pessoa"), F.col("nome_pessoa")]).alias("sk_person_id"), "nome_pessoa", "tipo_pessoa")
)
gravar_gold(dim_people, "gold.dim_people")

# ---------------- gold.dim_companies ----------------
dim_companies = (
    spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora")).distinct()
    .select(surrogate_key([F.col("nome_produtora")]).alias("sk_company_id"), "nome_produtora")
)
gravar_gold(dim_companies, "gold.dim_companies")

## Tabela Fato — `gold.fact_movies_performance`

In [ ]:
fin = spark.table("silver.tb_financeiro_filmes")
met = spark.table("silver.tb_metricas_engajamento")

fact = (
    spark.table("gold.dim_movies")
    .filter(F.col("status_filme") == "Lançado")                  # fato = filmes lançados
    .select("sk_movie_id", "id_filme")
    .join(fin, "id_filme", "left")                                # 1:1 (silver deduplicada)
    .join(met, "id_filme", "left")                                # 1:1 (silver deduplicada)
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)

# Garantia do grão: 1 linha por filme (joins não podem duplicar)
total, distintos = fact.count(), fact.select("sk_movie_id").distinct().count()
assert total == distintos, f"Grão violado: {total} linhas para {distintos} filmes"
gravar_gold(fact, "gold.fact_movies_performance")

## `gold.dim_reviews` (métrica resumida por filme)

In [ ]:
resumo_reviews = (
    spark.table("silver.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),                  # total de avaliações do filme
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"),  # média ignora notas NULL
    )
)

dim_reviews = (
    resumo_reviews
    .join(spark.table("gold.dim_movies").select("sk_movie_id", "id_filme"), "id_filme", "inner")  # só filmes do catálogo
    .select(surrogate_key([F.col("sk_movie_id")]).alias("sk_review_id"),
            "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)
gravar_gold(dim_reviews, "gold.dim_reviews")

## Tabelas-ponte (M:N) — evitam duplicar a fato

In [ ]:
mov = spark.table("gold.dim_movies").select("sk_movie_id", "id_filme")

# ---- bridge_movie_genre ----
bridge_movie_genre = (
    spark.table("silver.tb_generos")
    .join(mov, "id_filme", "inner")
    .join(spark.table("gold.dim_genres"), F.col("genero") == F.col("nome_genero"), "inner")
    .select("sk_movie_id", "sk_genre_id").distinct()
)
gravar_gold(bridge_movie_genre, "gold.bridge_movie_genre")

# ---- bridge_movie_person (chave composta nome + tipo) ----
bridge_movie_person = (
    spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(mov, "id_filme", "inner")
    .join(spark.table("gold.dim_people"),
          (F.col("nome_entidade") == F.col("nome_pessoa")) & (F.col("tipo_entidade") == F.col("tipo_pessoa")), "inner")
    .select("sk_movie_id", "sk_person_id").distinct()
)
gravar_gold(bridge_movie_person, "gold.bridge_movie_person")

# ---- bridge_movie_company ----
bridge_movie_company = (
    spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(mov, "id_filme", "inner")
    .join(spark.table("gold.dim_companies"), F.col("nome_entidade") == F.col("nome_produtora"), "inner")
    .select("sk_movie_id", "sk_company_id").distinct()
)
gravar_gold(bridge_movie_company, "gold.bridge_movie_company")

## Validação de integridade referencial (todas as contagens de órfãos devem ser 0)

In [ ]:
def orfaos(filha, col_fk, pai, col_pk):
    return spark.table(filha).join(spark.table(pai), spark.table(filha)[col_fk] == spark.table(pai)[col_pk], "left_anti").count()

verificacoes = [
    ("gold.fact_movies_performance", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    ("gold.dim_reviews", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    ("gold.bridge_movie_genre", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    ("gold.bridge_movie_genre", "sk_genre_id", "gold.dim_genres", "sk_genre_id"),
    ("gold.bridge_movie_person", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    ("gold.bridge_movie_person", "sk_person_id", "gold.dim_people", "sk_person_id"),
    ("gold.bridge_movie_company", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    ("gold.bridge_movie_company", "sk_company_id", "gold.dim_companies", "sk_company_id"),
]
for filha, fk, pai, pk in verificacoes:
    print(f"{filha}.{fk:14s} -> {pai}.{pk:14s} órfãos: {orfaos(filha, fk, pai, pk)}")